# Bagging & Random Forests

**Companion lesson:** https://ml-viz.vercel.app/courses/ensemble-methods/01-bagging-and-random-forests

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Bootstrap Aggregating

Each tree trains on a random sample with replacement. Averaging reduces variance.

In [ ]:
np.random.seed(42)
n = 100
X = np.sort(5 * np.random.rand(n))
y_true = np.sin(X) + 0.3 * np.random.randn(n)

def fit_stump(X, y):
    best_loss, best_t, best_v = np.inf, 0, 0
    for t in np.unique(X):
        for v in [y[X <= t].mean(), y[X > t].mean()]:
            pred = np.where(X <= t, y[X <= t].mean(), y[X > t].mean())
            loss = np.mean((y - pred)**2)
            if loss < best_loss:
                best_loss, best_t = loss, t
    return best_t

def predict_stump(x, X, y, t):
    return np.where(x <= t, y[X <= t].mean(), y[X > t].mean())

# Bootstrap samples
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x_grid = np.linspace(0, 5, 200)

for ax_i in range(3):
    ax = axes[ax_i]
    idx = np.random.choice(n, n, replace=True)
    t = fit_stump(X[idx], y_true[idx])
    pred = predict_stump(x_grid, X[idx], y_true[idx], t)
    ax.scatter(X, y_true, c='#818cf8', s=10, alpha=0.4)
    ax.plot(x_grid, pred, color='#f43f5e', linewidth=2)
    ax.set_title(f'Bootstrap Tree {ax_i+1}', color='white', fontsize=11)

plt.suptitle('Each Tree Sees Different Data', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Average of many trees
fig, ax = plt.subplots(figsize=(10, 5))
predictions = []
for _ in range(50):
    idx = np.random.choice(n, n, replace=True)
    t = fit_stump(X[idx], y_true[idx])
    pred = predict_stump(x_grid, X[idx], y_true[idx], t)
    predictions.append(pred)
    ax.plot(x_grid, pred, color='#818cf8', alpha=0.08, linewidth=0.5)

ensemble = np.mean(predictions, axis=0)
ax.plot(x_grid, np.sin(x_grid), color='#94a3b8', linestyle='--', linewidth=1.5, label='True function')
ax.plot(x_grid, ensemble, color='#14b8a6', linewidth=2.5, label='Ensemble (50 trees)')
ax.scatter(X, y_true, c='#818cf8', s=10, alpha=0.3)
ax.legend()
ax.set_title('Bagging: Averaging Reduces Variance', color='white')
plt.tight_layout()
plt.show()